# Polars 2.0 y DuckDB 2.0 frente a sus versiones 1.x

Cuaderno de lectura de la medición `polars-duckdb/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). TPC-H a escala 10 en Polars 1.43.1, 1.44.2 y 2.0.0 y en DuckDB 1.5.5, 1.5.6 y 2.0.0, el banco del 11 de agosto de 2026 repetido, tres agregaciones con 8 GiB de tope y la guía de migración de Polars 2.0 probada apartado a apartado. Carga los ficheros de al lado (o los descarga del repositorio si se ejecuta fuera de él), muestra la ficha de procedencia y dibuja una figura con matplotlib a secas. Solo lee; no repite la medición: para eso está `generar.py`.

In [ ]:
import urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/polars-duckdb/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")

## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El dato

In [ ]:
b = leer("b_tpch_resumen.csv")
b["configuracion"] = b.motor + " " + b.version.astype(str) + " " + b.modo
suma = b.groupby("configuracion")[["caliente_mediana_s", "fria_s"]].sum().round(2)
suma["pico_anon_mediano_mb"] = b.groupby("configuracion").pico_anon_mb_max.median().round(0)
print(suma.sort_values("caliente_mediana_s").to_string())
print()
d = leer("d_cambios_apartados.csv")
print("guía de migración de Polars 2.0:", len(d), "apartados")
print(d.clase.value_counts().to_string())

## Una figura

In [ ]:
t = b.pivot_table(index="consulta", columns="configuracion", values="caliente_mediana_s")
cols = ["polars 1.44.2 auto", "polars 2.0.0 auto", "duckdb 1.5.6 predeterminado"]
ax = t[cols].plot.bar(figsize=(10, 4), width=0.8)
ax.set_xlabel("consulta de TPC-H")
ax.set_ylabel("segundos en caliente")
ax.set_title("TPC-H a escala 10: Polars 1.44.2, Polars 2.0.0 y DuckDB 1.5.6")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()